In [1]:
# Explore dataset

import pandas as pd
data = pd.read_csv('D:\Real-Estate-AI-Assitant\data\cleaned_dataset.csv')

<>:4: SyntaxWarning: invalid escape sequence '\R'
<>:4: SyntaxWarning: invalid escape sequence '\R'
C:\Users\leona\AppData\Local\Temp\ipykernel_7228\3524964698.py:4: SyntaxWarning: invalid escape sequence '\R'
  data = pd.read_csv('D:\Real-Estate-AI-Assitant\data\cleaned_dataset.csv')


In [2]:
# Encode Categorical Variables

import sklearn.preprocessing as preprocessing
import sklearn.preprocessing as StandardScaler
label_encoder = preprocessing.LabelEncoder()
standard_scaler = StandardScaler.StandardScaler()

In [3]:
data['Lokasi']= label_encoder.fit_transform(data['Lokasi'])
data['Kota']= label_encoder.fit_transform(data['Kota'])


In [4]:
from sklearn.model_selection import train_test_split
Y = data['Harga']
X = data.drop('Harga', axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, shuffle=True)

In [5]:
# Validation Set Creation
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train, test_size=0.25, random_state=40, shuffle=True)  



In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
import scipy.stats as stats

X_train[['Luas Tanah', 'Luas Bangunan']] = standard_scaler.fit_transform(X_train[['Luas Tanah', 'Luas Bangunan']]) # Only fit on training data to avoid data leakage

# Hyperparameter Tuning
rf = RandomForestRegressor(random_state=42)

# Define the hyperparameter grid
param_dist = {
    'n_estimators': stats.randint(50, 500),
    'max_depth': stats.randint(3, 20),
    'max_features' : ['sqrt', 'log2', None],
    'min_samples_split': stats.randint(2, 20),
    'min_samples_leaf': stats.randint(1, 20),
    'bootstrap': [True, False]
}



In [ ]:
# Random Search CV
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = param_dist, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = 1, scoring='neg_mean_squared_error', error_score='raise')
rf_random.fit(X_train, y_train)

print("Best Hyperparameters:", rf_random.best_params_)
tuned_model = rf_random.best_estimator_

In [29]:
tuned_model.fit(X_train, y_train)
tuned_model.score(X_val, y_val)
print("Validation Set Score:", tuned_model.score(X_val, y_val))

#Test Data Evaluation
y_pred = tuned_model.predict(X_test)
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Test Set Mean Absolute Error:", mae)
print("Test Set R^2 Score:", r2)

Validation Set Score: 0.9984949992131809
Test Set Mean Absolute Error: 17368065.785330478
Test Set R^2 Score: 0.9966209970711106


In [28]:
# Check the scale of your predictions vs actual values
print("y_test range:", y_test.min(), "to", y_test.max())
print("y_pred range:", y_pred.min(), "to", y_pred.max())
print("Mean of y_test:", y_test.mean())
print("Mean of y_pred:", y_pred.mean())

# Look for extreme errors
errors = y_test - y_pred
print("Max error:", errors.max())
print("Min error:", errors.min())

y_test range: 300000000 to 9900000000
y_pred range: 300554166.4941838 to 9900000000.0
Mean of y_test: 2961407703.83992
Mean of y_pred: 2960906687.1330547
Max error: 1893208893.5330305
Min error: -2358787530.403557


In [30]:
import joblib
import os
from datetime import datetime

# Create a timestamp for versioning
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Path 
path = 'D:/Real-Estate-AI-Assitant/models/'

# Save the model
model_filename = os.path.join(path, f'rf_model_{timestamp}.pkl')
joblib.dump(tuned_model, model_filename)


# Save model metadata
metadata_filename = os.path.join(path, f'metadata_{timestamp}.pkl')
metadata = {
    'model_type': 'RandomForestRegressor',
    'mae': mae,
    'r2_score': r2,
    'train_date': timestamp,
    'feature_names': list(X_train.columns),  # if using DataFrame
    'best_params': tuned_model.get_params()
}

joblib.dump(metadata, metadata_filename)

print(f"Model saved as: {model_filename}")

Model saved as: D:/Real-Estate-AI-Assitant/models/rf_model_20260102_194601.pkl
